# 계절성지수

[데이터타입 통합]
kopo_channel_seasonality_new.csv 파일을 불러온 후
selloutData 변수에 담으세요 이후 
QTY컬럼→실수(float), 이외컬럼→문자(str)로 변경하세요

# 라이브러리 선언

In [16]:
import pandas as pd
import numpy as np

# 데이터 불러오기

In [17]:
selloutData=pd.read_csv('../dataset/kopo_channel_seasonality_new.csv')
selloutData

,REGIONID,PRODUCT,YEARWEEK,QTY
0,A60,PRODUCT4,201402,71.0
1,A60,PRODUCT59,201402,22275.0
2,A60,PRODUCT34,201402,4463.0
3,A60,PRODUCT47,201402,0.0
4,A60,PRODUCT56,201402,23.0
...,...,...,...,...
124653,A10,PRODUCT60,201630,824.0
124654,A10,PRODUCT56,201630,275.0
124655,A10,PRODUCT61,201630,0.0
124656,A10,PRODUCT12,201630,15021.0


# 데이터 타입 통합

In [18]:
# 모든 컬럼에 대해서 타입 고정 (데이터소스 변경에 따른 코드 영향성 제로 만들기)
selloutData.REGIONID = selloutData.REGIONID.astype(str)
selloutData.PRODUCT = selloutData.PRODUCT.astype(str)
selloutData.YEARWEEK = selloutData.YEARWEEK.astype(str)
selloutData.QTY = selloutData.QTY.astype(float)

In [19]:
selloutData

,REGIONID,PRODUCT,YEARWEEK,QTY
0,A60,PRODUCT4,201402,71.0
1,A60,PRODUCT59,201402,22275.0
2,A60,PRODUCT34,201402,4463.0
3,A60,PRODUCT47,201402,0.0
4,A60,PRODUCT56,201402,23.0
...,...,...,...,...
124653,A10,PRODUCT60,201630,824.0
124654,A10,PRODUCT56,201630,275.0
124655,A10,PRODUCT61,201630,0.0
124656,A10,PRODUCT12,201630,15021.0


[불량 데이터 처리]
kopo_channel_seasonality_new.csv 자료를 담은 
selloutData 변수에서
QTY컬럼 음수(반품)인 경우 0, 양수인 경우 기존 QTY 값
유지하는 로직을 적용하여 QTY_NEW 컬럼을 추가하세요

In [20]:
selloutData["QTY_NEW"] = np.where(selloutData.QTY < 0, 0, selloutData.QTY)
selloutData

,REGIONID,PRODUCT,YEARWEEK,QTY,QTY_NEW
0,A60,PRODUCT4,201402,71.0,71.0
1,A60,PRODUCT59,201402,22275.0,22275.0
2,A60,PRODUCT34,201402,4463.0,4463.0
3,A60,PRODUCT47,201402,0.0,0.0
4,A60,PRODUCT56,201402,23.0,23.0
...,...,...,...,...,...
124653,A10,PRODUCT60,201630,824.0,824.0
124654,A10,PRODUCT56,201630,275.0,275.0
124655,A10,PRODUCT61,201630,0.0,0.0
124656,A10,PRODUCT12,201630,15021.0,15021.0


In [21]:
selloutData.loc[selloutData.QTY_NEW < 0]

,REGIONID,PRODUCT,YEARWEEK,QTY,QTY_NEW


[데이터 통합]
selloutData 자료에서 
YEAR, WEEK 컬럼을 생성하고 WEEK 컬럼 값이 52 이하인
데이터만 refinedSelloutData 변수에 저장하세요

In [22]:
selloutData["YEAR"] = selloutData.YEARWEEK.str[:4].astype(int)
selloutData["WEEK"] = selloutData.YEARWEEK.str[4:].astype(int)

exceptWeek = 53
refinedSelloutData = selloutData[np.where(selloutData.WEEK < exceptWeek, True, False)]
refinedSelloutData

,REGIONID,PRODUCT,YEARWEEK,QTY,QTY_NEW,YEAR,WEEK
0,A60,PRODUCT4,201402,71.0,71.0,2014,2
1,A60,PRODUCT59,201402,22275.0,22275.0,2014,2
2,A60,PRODUCT34,201402,4463.0,4463.0,2014,2
3,A60,PRODUCT47,201402,0.0,0.0,2014,2
4,A60,PRODUCT56,201402,23.0,23.0,2014,2
...,...,...,...,...,...,...,...
124653,A10,PRODUCT60,201630,824.0,824.0,2016,30
124654,A10,PRODUCT56,201630,275.0,275.0,2016,30
124655,A10,PRODUCT61,201630,0.0,0.0,2016,30
124656,A10,PRODUCT12,201630,15021.0,15021.0,2016,30


In [23]:
refinedSelloutData.loc[refinedSelloutData.WEEK > 52]

,REGIONID,PRODUCT,YEARWEEK,QTY,QTY_NEW,YEAR,WEEK


In [24]:
selloutData.loc[selloutData.WEEK>52]

,REGIONID,PRODUCT,YEARWEEK,QTY,QTY_NEW,YEAR,WEEK
62075,A64,PRODUCT63,201553,2092.0,2092.0,2015,53
62076,A64,PRODUCT15,201553,80.0,80.0,2015,53
62077,A64,PRODUCT4,201553,1781.0,1781.0,2015,53
62078,A64,PRODUCT16,201553,8.0,8.0,2015,53
62079,A64,PRODUCT59,201553,24242.0,24242.0,2015,53
...,...,...,...,...,...,...,...
119082,A13,PRODUCT14,201553,729.0,729.0,2015,53
119083,A39,PRODUCT58,201553,269.0,269.0,2015,53
119084,A39,PRODUCT63,201553,581.0,581.0,2015,53
119085,A39,PRODUCT4,201553,290.0,290.0,2015,53


In [25]:
sortKey = ["REGIONID", "PRODUCT", "YEARWEEK"]
sortedData = refinedSelloutData.sort_values( by=sortKey ).reset_index(drop=True)

In [29]:
groupKey = ["REGIONID", "PRODUCT", "YEAR"]
groupData = sortedData.groupby(by=groupKey)["QTY_NEW"].agg(["mean"]).reset_index()
groupData = groupData.rename(columns={"mean" : "QTY_MEAN"})
groupData

,REGIONID,PRODUCT,YEAR,QTY_MEAN
0,A00,PRODUCT34,2014,275.961538
1,A00,PRODUCT34,2015,86.634615
2,A00,PRODUCT34,2016,36.576923
3,A00,PRODUCT58,2014,2.673077
4,A00,PRODUCT58,2015,5.711538
...,...,...,...,...
2377,A77,PRODUCT1,2015,3030.019231
2378,A77,PRODUCT1,2016,3375.326923
2379,A77,PRODUCT12,2014,2035.788462
2380,A77,PRODUCT12,2015,3540.980769


In [43]:
mergeKey = ["REGIONID", "PRODUCT", "YEAR"]
mergedData = pd.merge(left=refinedSelloutData, right=groupData, on=mergeKey, how="left")
mergedData = mergedData.sort_values(["REGIONID", "PRODUCT", "YEARWEEK"], ascending=True)
mergedData

,REGIONID,PRODUCT,YEARWEEK,QTY,QTY_NEW,YEAR,WEEK,QTY_MEAN
298,A00,PRODUCT34,201401,661.0,661.0,2014,1,275.961538
1757,A00,PRODUCT34,201402,679.0,679.0,2014,2,275.961538
3125,A00,PRODUCT34,201403,578.0,578.0,2014,3,275.961538
205,A00,PRODUCT34,201404,532.0,532.0,2014,4,275.961538
4369,A00,PRODUCT34,201405,516.0,516.0,2014,5,275.961538
...,...,...,...,...,...,...,...,...
101633,A77,PRODUCT12,201648,4152.0,4152.0,2016,48,4837.153846
96866,A77,PRODUCT12,201649,5086.0,5086.0,2016,49,4837.153846
101670,A77,PRODUCT12,201650,5846.0,5846.0,2016,50,4837.153846
100552,A77,PRODUCT12,201651,4933.0,4933.0,2016,51,4837.153846


In [44]:
mergedData['SEASONALITY'] = mergedData['QTY_NEW'] / mergedData['QTY_MEAN']
mergedData

,REGIONID,PRODUCT,YEARWEEK,QTY,QTY_NEW,YEAR,WEEK,QTY_MEAN,SEASONALITY
298,A00,PRODUCT34,201401,661.0,661.0,2014,1,275.961538,2.395261
1757,A00,PRODUCT34,201402,679.0,679.0,2014,2,275.961538,2.460488
3125,A00,PRODUCT34,201403,578.0,578.0,2014,3,275.961538,2.094495
205,A00,PRODUCT34,201404,532.0,532.0,2014,4,275.961538,1.927805
4369,A00,PRODUCT34,201405,516.0,516.0,2014,5,275.961538,1.869826
...,...,...,...,...,...,...,...,...,...
101633,A77,PRODUCT12,201648,4152.0,4152.0,2016,48,4837.153846,0.858356
96866,A77,PRODUCT12,201649,5086.0,5086.0,2016,49,4837.153846,1.051445
101670,A77,PRODUCT12,201650,5846.0,5846.0,2016,50,4837.153846,1.208562
100552,A77,PRODUCT12,201651,4933.0,4933.0,2016,51,4837.153846,1.019815


In [58]:
finData = ["REGIONID", "PRODUCT", "WEEK"]
finalResult = mergedData.groupby(by=finData)["SEASONALITY"].mean().reset_index()
finalResult = finalResult.rename(columns={"REGIONID":"regionid", "PRODUCT":"product", "WEEK":"week", "SEASONALITY":"seasonality"})

In [59]:
finalResult

,regionid,product,week,seasonality
0,A00,PRODUCT34,1,1.570782
1,A00,PRODUCT34,2,1.755540
2,A00,PRODUCT34,3,1.319460
3,A00,PRODUCT34,4,1.490298
4,A00,PRODUCT34,5,1.061909
...,...,...,...,...
41283,A77,PRODUCT12,48,1.352712
41284,A77,PRODUCT12,49,1.094083
41285,A77,PRODUCT12,50,1.386116
41286,A77,PRODUCT12,51,1.255192
